In [1]:
import pandas as pd 
import os 
import glob 
import time 
import logging

In [2]:
DATA_DIR= "../../data/raw"
OUTPUT_FILE = "../../data/dataset.csv"
LOG_DIR ='../../logs'
LOG_FILE= os.path.join(LOG_DIR,'parquet_to_csv.log')
os.makedirs(LOG_DIR,exist_ok=True)

In [3]:
logger = logging.getLogger("ParquetToCSVLogger")
logger.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

file_handler = logging.FileHandler(LOG_FILE)
file_handler.setFormatter(formatter)

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

In [5]:
import psutil

In [6]:
start_time = time.time()
process = psutil.Process()
all_dfs = []
total_rows = 0

logger.info("Starting conversion from Parquet to CSV")
logger.info(f"Looking in: {DATA_DIR}")

2025-12-26 15:15:20,378 | INFO | Starting conversion from Parquet to CSV
2025-12-26 15:15:20,379 | INFO | Looking in: ../../data/raw


In [7]:
try:
    partitions = [os.path.join(DATA_DIR, d) for d in os.listdir(DATA_DIR) if d.startswith("year_month=")]
    for partition in partitions:
        files = glob.glob(os.path.join(partition, "*.parquet"))
        logger.info(f"Found {len(files)} files in {partition}")

        for f in files:
            df = pd.read_parquet(f)
            all_dfs.append(df)
            total_rows += len(df)
            mem_mb = process.memory_info().rss / (1024 * 1024)
            logger.info(f"Loaded {len(df)} rows from {f} | Memory usage: {mem_mb:.2f} MB")

    # Combine all data
    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df.to_csv(OUTPUT_FILE, index=False)
    duration = time.time() - start_time
    mem_mb = process.memory_info().rss / (1024 * 1024)

    logger.info(f"CSV saved to {OUTPUT_FILE}")
    logger.info(f"Total rows: {total_rows}")
    logger.info(f"Total time taken: {duration:.2f} seconds")
    logger.info(f"Final memory usage: {mem_mb:.2f} MB")

except MemoryError:
    logger.error("MemoryError: Dataset too large to fit in memory for Pandas")
except Exception as e:
    logger.error("An error occurred", exc_info=True)

2025-12-26 15:15:48,427 | INFO | Found 1 files in ../../data/raw\year_month=2024-02
2025-12-26 15:15:49,544 | INFO | Loaded 100000 rows from ../../data/raw\year_month=2024-02\transactions_b51f6958b9dd4c9cbb09bc848d3eba23.parquet | Memory usage: 257.20 MB
2025-12-26 15:15:49,546 | INFO | Found 3 files in ../../data/raw\year_month=2024-03
2025-12-26 15:15:49,697 | INFO | Loaded 100000 rows from ../../data/raw\year_month=2024-03\transactions_006240cc6d3d4c19b7242067e65ba264.parquet | Memory usage: 314.45 MB
2025-12-26 15:15:49,853 | INFO | Loaded 100000 rows from ../../data/raw\year_month=2024-03\transactions_658caf20eca341428a103cfb9f73f23c.parquet | Memory usage: 360.91 MB
2025-12-26 15:15:50,017 | INFO | Loaded 100000 rows from ../../data/raw\year_month=2024-03\transactions_ac1597b660e9464e8ce7c86ead163d83.parquet | Memory usage: 413.34 MB
2025-12-26 15:15:50,019 | INFO | Found 1 files in ../../data/raw\year_month=2024-05
2025-12-26 15:15:50,184 | INFO | Loaded 100000 rows from ../../d